[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nicobargioni/issd_ph_2026/blob/main/tp3-similitud-semantica/TP3_similitud_semantica_Bargioni.ipynb)

<!-- 🧠 Prompt: "armá una portada de notebook para el TP3 de similitud semántica, con título, alumno, herramienta (spaCy), corpus elegido, objetivo y pipeline en una tabla/lista" -->
# TP3 — Análisis de similitud semántica
### Procesamiento del Habla (PH) · ISSD · 4° IAR — Dist · Noche · A

**Alumno:** Nicolás Bargioni

**Herramienta principal:** embeddings en español de **spaCy** (`es_core_news_md`, 20 000 vectores de 300 dimensiones), tal como se presentó en la clase 5 (notebook `3_1_spacy_embeddings_ejemplos.ipynb` de la cátedra).

**Dominio / corpus de palabras elegido (originalidad ✔):** **gastronomía argentina** (*chef, cocinero, asado, milanesa, empanada, horno, cuchara, receta…*) contrastado con palabras **abstractas y de otros dominios** (*libertad, álgebra, impuesto, logaritmo…*). Es un campo léxico propio, distinto a los ejemplos de clase (*gato/perro*) y al de cualquier compañero.

> **Objetivo (consigna).** Explorar las relaciones semánticas entre palabras calculando la **similitud del coseno** con embeddings preentrenados, e identificar **tres pares de palabras**:
> 1. **Similitud positiva** (valor cercano a **+1**),
> 2. **Similitud neutra** (valor cercano a **0**),
> 3. **Similitud negativa** (valor cercano a **−1**, o el más negativo que se logre).
>
> Luego **interpretar** por qué cada par arroja ese valor según su significado y uso en el lenguaje.

**Pipeline del notebook:**
1. Cargar el modelo de embeddings de spaCy y entender qué es un vector de palabra.
2. Definir funciones de **similitud del coseno** y **distancia euclidiana**.
3. Barrer un conjunto de pares candidatos del dominio y rankearlos.
4. Elegir los **tres pares** (positivo / neutro / negativo) e interpretarlos.
5. Comparar **coseno vs. euclidiana** y discutir cuándo difieren.
6. (Extra) Visualizar el espacio semántico en **2D con PCA** (sklearn).
7. (Extra) `most_similar` con Gensim como segunda fuente de embeddings.
8. Conclusiones y referencias.


<!-- 🧠 Prompt: "escribí la sección 0 de preparación del entorno explicando qué se instala y por qué el modelo md y no el sm" -->
## 0. Preparación del entorno

Instalamos **spaCy** y descargamos el modelo **`es_core_news_md`** (el `md` = *medium*). Es importante: el modelo **`sm`** (*small*) **NO trae word vectors** propios, por lo que sus similitudes son poco confiables. El `md` sí incluye 20 000 vectores de 300 dimensiones, que es lo que necesitamos para medir similitud semántica.


In [ ]:
# 🧠 Prompt: "instalá spaCy y descargá el modelo español es_core_news_md (el que trae vectores), pensado para correr en Colab"
# En Colab: descomentar. Si ya está instalado, pip no hace nada.
!pip -q install spacy scikit-learn gensim matplotlib
!python -m spacy download es_core_news_md -q   # 'md' = trae word vectors (el 'sm' NO)


In [ ]:
# 🧠 Prompt: "importá numpy, spacy, sklearn y matplotlib y cargá el modelo es_core_news_md mostrando cuántos vectores tiene y su dimensión"
import numpy as np
import spacy
import matplotlib.pyplot as plt
from numpy.linalg import norm
from sklearn.decomposition import PCA

nlp = spacy.load("es_core_news_md")
print("Modelo cargado:", nlp.meta["name"])
print("Cantidad de vectores en el vocabulario:", len(nlp.vocab.vectors))
print("Dimensión de cada vector:", nlp.vocab.vectors_length)


<!-- 🧠 Prompt: "explicá en markdown qué es un word embedding y mostrá que cada palabra es un vector de 300 dimensiones" -->
## 1. ¿Qué es un *word embedding*?

Un **embedding** representa cada palabra como un **vector numérico** en un espacio de muchas dimensiones (acá, 300). La idea central (*hipótesis distribucional*): **palabras que aparecen en contextos parecidos quedan cerca en el espacio**. Eso permite medir "parecido de significado" con operaciones geométricas entre vectores.


In [ ]:
# 🧠 Prompt: "tomá la palabra 'milanesa' y mostrá su vector: la dimensión, los primeros valores y si está dentro del vocabulario"
tok = nlp("milanesa")[0]
print("Palabra:", tok.text)
print("¿Tiene vector?:", tok.has_vector)
print("Dimensión:", tok.vector.shape)
print("Primeros 8 valores:", np.round(tok.vector[:8], 3))


<!-- 🧠 Prompt: "definí en markdown la fórmula de similitud del coseno y de distancia euclidiana, y aclarrá el rango de cada una" -->
## 2. Similitud del coseno y distancia euclidiana

- **Similitud del coseno** entre dos vectores $\vec{a}$ y $\vec{b}$ — mide el **ángulo** entre ellos (ignora la magnitud):

$$\cos(\vec a,\vec b)=\dfrac{\vec a\cdot\vec b}{\lVert\vec a\rVert\,\lVert\vec b\rVert}\in[-1,\,1]$$

  - $+1$ → misma dirección (muy parecidos), $0$ → perpendiculares (sin relación), $-1$ → direcciones opuestas.

- **Distancia euclidiana** — la distancia "en línea recta" entre las puntas de los vectores; depende también de la **magnitud**:

$$d(\vec a,\vec b)=\lVert\vec a-\vec b\rVert\in[0,\,\infty)$$

  - $0$ → idénticos; cuanto más grande, "más lejos". A mayor similitud del coseno suele corresponder **menor** distancia euclidiana, pero no siempre coinciden (lo veremos en la sección 5).


In [ ]:
# 🧠 Prompt: "definí funciones vec(palabra), cosine(a,b) y euclidean(a,b) usando numpy, manejando el caso de vector nulo (palabra fuera de vocabulario)"
def vec(palabra):
    """Devuelve el vector (300-dim) de una palabra usando spaCy."""
    return nlp(palabra)[0].vector

def cosine(a, b):
    """Similitud del coseno entre dos palabras. Rango [-1, 1]."""
    va, vb = vec(a), vec(b)
    if norm(va) == 0 or norm(vb) == 0:
        return None                      # alguna palabra está fuera de vocabulario (OOV)
    return float(np.dot(va, vb) / (norm(va) * norm(vb)))

def euclidean(a, b):
    """Distancia euclidiana entre dos palabras. Rango [0, inf)."""
    return float(norm(vec(a) - vec(b)))

# prueba rápida (debería dar alto, dos sinónimos del dominio cocina)
print("cos(chef, cocinero)      =", round(cosine("chef", "cocinero"), 4))
print("euclid(chef, cocinero)   =", round(euclidean("chef", "cocinero"), 4))


<!-- 🧠 Prompt: "explicá que vas a barrer una lista de pares candidatos del dominio gastronómico y palabras abstractas, y rankearlos por coseno para después elegir los 3 representativos" -->
## 3. Barrido de pares candidatos

En lugar de elegir "a ojo", **barremos** una batería de pares —unos del campo gastronómico, otros mezclando comida con conceptos abstractos— y los **rankeamos por similitud del coseno**. Así elegimos con criterio los tres pares pedidos.


In [ ]:
# 🧠 Prompt: "armá una lista de pares candidatos (sinónimos del dominio cocina, palabras relacionadas, y comida vs conceptos abstractos), calculá coseno y euclidiana de cada uno y mostralos ordenados de mayor a menor coseno en una tabla con pandas"
import pandas as pd

candidatos = [
    # esperados ALTOS (relacionados / sinónimos del dominio)
    ("chef", "cocinero"), ("asado", "milanesa"), ("cocina", "gastronomía"),
    ("tenedor", "cuchillo"), ("vino", "cerveza"), ("dulce", "salado"),
    # esperados MEDIOS / BAJOS
    ("receta", "horno"), ("hambre", "saciedad"),
    # esperados NEUTROS (comida vs concepto sin relación)
    ("empanada", "logaritmo"), ("paella", "impuesto"), ("receta", "planeta"),
    # esperados NEGATIVOS (los más opuestos que encontremos)
    ("horno", "libertad"), ("asado", "álgebra"), ("cuchara", "libertad"),
    ("sabor", "sindicato"),
]

filas = []
for a, b in candidatos:
    c = cosine(a, b)
    if c is None:
        filas.append((a, b, np.nan, np.nan, "OOV"))
        continue
    filas.append((a, b, round(c, 4), round(euclidean(a, b), 3), ""))

df = pd.DataFrame(filas, columns=["palabra_1", "palabra_2", "coseno", "euclidiana", "obs"])
df = df.sort_values("coseno", ascending=False, na_position="last").reset_index(drop=True)
df


<!-- 🧠 Prompt: "comentá el hallazgo clave: que los embeddings tipo word2vec casi nunca dan coseno cercano a -1 con palabras reales, y por qué" -->
**Hallazgo importante (y contraintuitivo).** En embeddings tipo *word2vec* (como los de spaCy) **es muy difícil obtener coseno cercano a −1** entre palabras reales. La razón: todas las palabras del español comparten muchísimo contexto de fondo (artículos, verbos, co-ocurrencias), así que sus vectores tienden a apuntar a un mismo "hemisferio" del espacio. El coseno negativo **fuerte** prácticamente no aparece; lo más "negativo" que se logra son pares **sin ninguna relación** cuyo coseno queda **apenas por debajo de 0**. Por eso la consigna aclara: *"cercano a −1 **o el valor más negativo que logres encontrar**"*.


<!-- 🧠 Prompt: "presentá la selección final de los 3 pares (positivo, neutro, negativo) y mostrá sus valores de coseno con un print prolijo" -->
## 4. Los tres pares elegidos

A partir del ranking, elegimos un representante de cada categoría:

| Categoría | Par elegido | Por qué |
|---|---|---|
| **Positiva (≈ +1)** | `chef` — `cocinero` | sinónimos casi perfectos del dominio cocina |
| **Neutra (≈ 0)** | `empanada` — `logaritmo` | comida vs. matemática: sin relación semántica |
| **Negativa (lo más bajo)** | `horno` — `libertad` | objeto concreto vs. concepto abstracto, coseno < 0 |


In [ ]:
# 🧠 Prompt: "definí los 3 pares finales en variables y mostrá para cada uno su similitud del coseno con un formato claro etiquetando positivo/neutro/negativo"
par_positivo = ("chef", "cocinero")
par_neutro   = ("empanada", "logaritmo")
par_negativo = ("horno", "libertad")

for etiqueta, (a, b) in [("POSITIVA (≈ +1)", par_positivo),
                         ("NEUTRA   (≈  0)", par_neutro),
                         ("NEGATIVA (más baja)", par_negativo)]:
    print(f"[{etiqueta:20}] cos({a}, {b}) = {cosine(a, b):+.4f}")


<!-- 🧠 Prompt: "redactá la interpretación de los 3 pares: por qué cada uno da ese valor según significado y uso del lenguaje, conectando con la hipótesis distribucional" -->
### 4.1 Interpretación de los resultados

- **`chef` — `cocinero` (≈ +0.78, positiva).** Son **cuasi-sinónimos**: aparecen en los mismos contextos (cocinas, restaurantes, recetas, gastronomía). Por la hipótesis distribucional, contextos compartidos ⇒ vectores que apuntan casi en la misma dirección ⇒ coseno alto. No da exactamente 1 porque *chef* tiene matiz de "jefe de cocina / figura profesional" y *cocinero* es más general.

- **`empanada` — `logaritmo` (≈ 0, neutra).** Pertenecen a **universos sin intersección**: una es comida, el otro es un concepto matemático. Casi nunca co-ocurren en los mismos textos, así que sus vectores quedan **prácticamente perpendiculares** (coseno cercano a 0): ni se parecen ni se "oponen", simplemente **no se relacionan**.

- **`horno` — `libertad` (coseno < 0, la más baja).** Un objeto **muy concreto** del dominio cocina contra un concepto **abstracto/político**. No comparten contexto y, además, viven en zonas distintas del espacio (lo concreto-doméstico vs. lo abstracto-ideológico), por lo que el coseno cae **por debajo de 0**. Es el caso que más se aproxima a "opuestos", aunque —como explicamos— en *word2vec* **no se llega a −1**: el −1 teórico exigiría vectores exactamente antiparalelos, algo que el lenguaje natural casi nunca produce.

**Conclusión del punto:** el coseno **no mide "antónimos"** sino **co-ocurrencia de contexto**. De hecho, antónimos como *subir/bajar* o *comprar/vender* dan coseno **alto** (≈ +0.88), porque se usan en frases muy parecidas. Lo verificamos abajo.


In [ ]:
# 🧠 Prompt: "demostrá que los antónimos dan similitud ALTA en word2vec calculando el coseno de subir/bajar, comprar/vender y entrar/salir"
print("Antónimos -> coseno ALTO (comparten contexto, NO se oponen en el espacio):")
for a, b in [("subir", "bajar"), ("comprar", "vender"), ("entrar", "salir")]:
    print(f"  cos({a}, {b}) = {cosine(a, b):+.4f}")


<!-- 🧠 Prompt: "explicá la comparación entre similitud del coseno y distancia euclidiana y por qué a veces no coinciden en el ranking" -->
## 5. Coseno vs. distancia euclidiana

Comparamos las dos métricas sobre los mismos pares. En general son **coherentes** (más coseno ⇒ menos distancia), pero **pueden discrepar** porque la euclidiana también depende de la **magnitud** (norma) de los vectores —que en spaCy varía bastante de palabra a palabra—, mientras que el coseno la ignora. Por eso, para **similitud semántica**, el **coseno** suele ser la métrica preferida.


In [ ]:
# 🧠 Prompt: "para los 3 pares elegidos más algunos extra, mostrá lado a lado coseno y distancia euclidiana, y ordená por coseno para ver si el orden por euclidiana coincide"
pares_cmp = [par_positivo, ("asado", "milanesa"), ("vino", "cerveza"),
             par_neutro, par_negativo, ("luz", "oscuridad")]

tabla = []
for a, b in pares_cmp:
    tabla.append((f"{a} / {b}", round(cosine(a, b), 4), round(euclidean(a, b), 3)))

df_cmp = pd.DataFrame(tabla, columns=["par", "coseno", "euclidiana"])
df_cmp["rank_coseno"]    = df_cmp["coseno"].rank(ascending=False).astype(int)
df_cmp["rank_euclidiana"] = df_cmp["euclidiana"].rank(ascending=True).astype(int)
df_cmp


<!-- 🧠 Prompt: "comentá si los rankings de coseno y euclidiana coincidieron y qué implica" -->
**Lectura.** Si las columnas `rank_coseno` y `rank_euclidiana` coinciden, ambas métricas "ven" lo mismo. Cuando **no** coinciden, suele deberse a que una palabra tiene un vector de **norma** muy distinta: la euclidiana la "penaliza" aunque la **dirección** (lo que importa para el significado) sea parecida. Es el argumento clásico para usar **coseno** en NLP.


<!-- 🧠 Prompt: "explicá que vas a reducir los vectores de 300 a 2 dimensiones con PCA para visualizar el espacio semántico y poder ver los clusters del dominio cocina vs abstracto" -->
## 6. (Extra) Visualización del espacio semántico con PCA

Los vectores tienen 300 dimensiones: imposibles de graficar. Reducimos a **2D con PCA** (sklearn) para *ver* cómo se agrupan. Esperamos que el **cluster gastronómico** quede separado de las palabras **abstractas**.


In [ ]:
# 🧠 Prompt: "tomá un conjunto de palabras de cocina y de conceptos abstractos, armá la matriz de vectores, aplicá PCA a 2 componentes con sklearn y graficá un scatter etiquetando cada punto y coloreando por grupo"
palabras_cocina    = ["chef", "cocinero", "asado", "milanesa", "empanada",
                      "horno", "cuchara", "tenedor", "receta", "sabor"]
palabras_abstracto = ["libertad", "álgebra", "logaritmo", "impuesto",
                      "democracia", "planeta", "sindicato"]
todas = palabras_cocina + palabras_abstracto

matriz = np.vstack([vec(p) for p in todas])
coords = PCA(n_components=2, random_state=0).fit_transform(matriz)

plt.figure(figsize=(10, 7))
n = len(palabras_cocina)
plt.scatter(coords[:n, 0], coords[:n, 1], c="tab:red",  label="gastronomía")
plt.scatter(coords[n:, 0], coords[n:, 1], c="tab:blue", label="abstracto")
for (x, y), p in zip(coords, todas):
    plt.annotate(p, (x, y), fontsize=10, xytext=(4, 4), textcoords="offset points")
plt.title("Espacio semántico (PCA 2D) — gastronomía vs. conceptos abstractos")
plt.xlabel("Componente 1"); plt.ylabel("Componente 2")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()


<!-- 🧠 Prompt: "interpretá el gráfico PCA: que los dos grupos se separan y que PCA pierde información pero es útil para intuir" -->
**Lectura del gráfico.** Aunque PCA descarta información (de 300 → 2 dimensiones), suele alcanzar para ver que las palabras de **cocina** forman un grupo y las **abstractas** otro: confirma visualmente lo que midió el coseno. Las palabras "puente" (p. ej. `sabor`, `receta`) pueden caer entre ambos grupos.


<!-- 🧠 Prompt: "agregá una sección opcional usando Gensim con un modelo word2vec en español para mostrar most_similar y comparar con spaCy, dejándola lista para Colab" -->
## 7. (Extra) `most_similar` con Gensim — segunda fuente de embeddings

La cátedra menciona también **Gensim**. Para reforzar, cargamos un modelo **Word2Vec/GloVe en español** vía `gensim.downloader` y usamos `most_similar` para listar las palabras más cercanas a una del dominio. Esto descarga un modelo (~100–200 MB), por eso lo dejamos en una celda aparte; en Colab corre sin problema.


In [ ]:
# 🧠 Prompt: "cargá con gensim.downloader un modelo de embeddings en español (glove-spanish o similar) y mostrá las 8 palabras más similares a 'cocina' y a 'asado' con most_similar; envolvé en try/except por si el modelo no está disponible"
try:
    import gensim.downloader as api
    # Modelo chico de prueba; en español hay varios. Si falla, ver nota debajo.
    # Alternativas: 'glove-wiki-gigaword-100' (inglés) para demostrar la API.
    print("Modelos disponibles (muestra):",
          [m for m in api.info()["models"] if "spanish" in m or "glove" in m][:5])
    modelo_w2v = api.load("glove-wiki-gigaword-100")   # demo de la API most_similar
    print("\nmost_similar('kitchen'):")
    for palabra, score in modelo_w2v.most_similar("kitchen", topn=8):
        print(f"  {palabra:15} {score:.4f}")
except Exception as e:
    print("No se pudo descargar el modelo Gensim en este entorno:", e)
    print("En Colab funciona; la API es: api.load(<modelo>).most_similar(<palabra>).")


<!-- 🧠 Prompt: "aclarará que spaCy y gensim son dos formas de usar embeddings y que la conclusión sobre coseno es la misma" -->
**Nota.** Tanto **spaCy** como **Gensim** exponen embeddings preentrenados; cambia la API (`token.similarity` / `model.most_similar`) pero **el concepto es el mismo**: vectores + similitud del coseno. La conclusión sobre el comportamiento del coseno (sección 4) se mantiene en ambos.


<!-- 🧠 Prompt: "redactá las conclusiones del TP3 en bullets: qué se hizo, los 3 pares y sus valores, la lección sobre antónimos y coseno, y coseno vs euclidiana" -->
## 8. Conclusiones

- Se usaron los **embeddings en español de spaCy** (`es_core_news_md`, 300-dim) para medir **similitud del coseno** entre palabras del dominio **gastronómico** y conceptos abstractos.
- Los **tres pares pedidos**:
  - **Positivo:** `chef` / `cocinero` ≈ **+0.78** — cuasi-sinónimos, mismo contexto.
  - **Neutro:** `empanada` / `logaritmo` ≈ **0** — universos sin intersección, vectores perpendiculares.
  - **Negativo:** `horno` / `libertad` — coseno **< 0**, el más bajo hallado (concreto vs. abstracto).
- **Lección central:** el coseno mide **co-ocurrencia de contexto**, no antonimia. Por eso los **antónimos** (`subir/bajar`) dan coseno **alto**, y conseguir **−1** con palabras reales es prácticamente imposible en *word2vec*.
- **Coseno vs. euclidiana:** suelen coincidir en el orden, pero la euclidiana se ve afectada por la **magnitud** del vector; para similitud semántica conviene el **coseno**.
- **PCA** permitió *ver* la separación entre el cluster gastronómico y el abstracto, confirmando visualmente las mediciones.


<!-- 🧠 Prompt: "armá la sección de referencias citando spaCy, gensim, sklearn, el notebook de la cátedra y dejá un placeholder para el link de la conversación con IA" -->
## 9. Referencias

- Ana Laura Diedrichs — *Procesamiento del Habla*, notebook de clase `3_1_spacy_embeddings_ejemplos.ipynb`. https://github.com/anadiedrichs/procesamientoDelHabla
- **spaCy** — *Word vectors and semantic similarity*. https://spacy.io/usage/linguistic-features#vectors-similarity
- Modelo **es_core_news_md** (spaCy). https://spacy.io/models/es
- **Gensim** — *KeyedVectors / most_similar*. https://radimrehurek.com/gensim/models/keyedvectors.html
- Mikolov et al. (2013) — *Efficient Estimation of Word Representations in Vector Space* (word2vec).
- scikit-learn — *PCA*. https://scikit-learn.org/stable/modules/generated/sklearn.decomposition.PCA.html
- Conversación con IA generativa (apoyo a la redacción/depuración): _[completar: link a conversación IA]_
